# Transformações da Camada Silver (Dados Brutos para Dados Limpos)  

## 🎯 Objetivo 
Implementar transformações da camada Silver com **SCD Tipo 2** para rastreamento histórico de mudanças, aplicando regras de negócio rigorosas.

### 📋 Tarefas Implementadas
- ✅ Ingerir dados da camada Bronze com controle de qualidade
- ✅ Aplicar validações e regras de negócio específicas da Olist
- ✅ Implementar SCD Tipo 2 para tracking de mudanças históricas
- ✅ Padronizar tipos de dados e formatos
- ✅ Criar colunas derivadas para análise temporal
- ✅ Aplicar data quality checks e governança
- ✅ Salvar com versionamento e auditoria

### 🔄 SCD Tipo 2 - Slowly Changing Dimensions
Esta implementação rastreia mudanças históricas usando:
- **effective_date**: Data de início da versão do registro
- **end_date**: Data de fim da versão (NULL para atual)
- **is_current**: Flag indicando se é a versão atual
- **record_hash**: Hash para detectar mudanças nos dados
- **change_reason**: Motivo da mudança (INSERT, UPDATE, DATA_CORRECTION)



## 📚 Importação das Bibliotecas
Bibliotecas otimizadas para SCD Tipo 2 e processamento de dados de alta qualidade.

## ⚡ SPARK SESSION 
Configuração otimizada para processamento SCD e Delta Lake.

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, lit, when, isnan, isnull, 
    current_timestamp, current_date,
    year, month, dayofmonth, dayofweek,
    regexp_replace, trim, upper, lower,
    md5, concat_ws, sha2,
    lead, lag, row_number, rank, dense_rank,
    max as spark_max, min as spark_min,
    sum as spark_sum, count as spark_count,
    avg as spark_avg
)
import hashlib
from datetime import datetime, date


StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 3, Finished, Available, Finished)

In [ ]:
# Configuração otimizada do Spark para SCD Tipo 2 e Delta Lake
spark = SparkSession.builder.appName("SilverTransformations-SCD2-Olist").getOrCreate()

# Configurações de otimização Delta Lake
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")

# Configurações para SCD Tipo 2
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")

# Configurações de Data Quality
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("✅ Spark Session configurada para SCD Tipo 2 e processamento otimizado")

StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 4, Finished, Available, Finished)

## 🔧 Funções Utilitárias para SCD Tipo 2
Implementação de funções reutilizáveis para aplicar SCD Tipo 2 nas transformações.

In [ ]:
def add_scd2_columns(df, business_key_cols, exclude_cols=None):
    """
    Adiciona colunas necessárias para SCD Tipo 2
    
    Args:
        df: DataFrame de entrada
        business_key_cols: Lista de colunas que compõem a chave de negócio
        exclude_cols: Colunas a excluir do cálculo do hash (ex: timestamps de sistema)
    
    Returns:
        DataFrame com colunas SCD2 adicionadas
    """
    if exclude_cols is None:
        exclude_cols = []
    
    # Colunas para hash (excluindo metadados e timestamps de sistema)
    hash_cols = [c for c in df.columns if c not in exclude_cols + ['effective_date', 'end_date', 'is_current', 'record_hash', 'change_reason']]
    
    return df.withColumn("effective_date", current_date()) \
             .withColumn("end_date", lit(None).cast(DateType())) \
             .withColumn("is_current", lit(True)) \
             .withColumn("record_hash", sha2(concat_ws("|", *[coalesce(col(c), lit("NULL")) for c in hash_cols]), 256)) \
             .withColumn("change_reason", lit("INITIAL_LOAD")) \
             .withColumn("created_timestamp", current_timestamp()) \
             .withColumn("updated_timestamp", current_timestamp())

def apply_business_rules_customers(df):
    """
    Aplica regras de negócio específicas para clientes Olist
    """
    return df.withColumn("customer_zip_code_prefix", 
                        when(col("customer_zip_code_prefix").rlike("^[0-9]{5}$"), 
                             col("customer_zip_code_prefix"))
                        .otherwise("00000")) \
             .withColumn("customer_city", 
                        regexp_replace(trim(upper(col("customer_city"))), "[^A-Z0-9 ]", "")) \
             .withColumn("customer_state", 
                        when(col("customer_state").rlike("^[A-Z]{2}$"), 
                             upper(trim(col("customer_state"))))
                        .otherwise("XX")) \
             .withColumn("is_valid_customer", 
                        when((col("customer_id").isNotNull()) & 
                             (col("customer_unique_id").isNotNull()) &
                             (col("customer_zip_code_prefix") != "00000") &
                             (col("customer_state") != "XX"), True)
                        .otherwise(False))

def apply_business_rules_orders(df):
    """
    Aplica regras de negócio específicas para pedidos Olist
    """
    return df.withColumn("order_status", 
                        when(col("order_status").isin(["delivered", "shipped", "processing", "approved", "invoiced", "canceled", "unavailable"]), 
                             lower(trim(col("order_status"))))
                        .otherwise("unknown")) \
             .withColumn("is_completed_order", 
                        when(col("order_status").isin(["delivered", "shipped"]), True)
                        .otherwise(False)) \
             .withColumn("delivery_days", 
                        when(col("order_delivered_customer_date").isNotNull() & 
                             col("order_purchase_timestamp").isNotNull(),
                             datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
                        .otherwise(None)) \
             .withColumn("is_on_time_delivery",
                        when(col("order_delivered_customer_date") <= col("order_estimated_delivery_date"), True)
                        .when(col("order_delivered_customer_date").isNull(), None)
                        .otherwise(False))

def apply_data_quality_checks(df, table_name):
    """
    Aplica verificações de qualidade de dados
    """
    total_records = df.count()
    
    # Contagem de registros com problemas
    null_counts = {}
    for column in df.columns:
        null_count = df.filter(col(column).isNull()).count()
        if null_count > 0:
            null_counts[column] = null_count
    
    # Log de qualidade de dados
    print(f"📊 Data Quality Report - {table_name}")
    print(f"   Total Records: {total_records:,}")
    
    if null_counts:
        print(f"   Null Values Found:")
        for col_name, null_count in null_counts.items():
            percentage = (null_count / total_records) * 100
            print(f"     - {col_name}: {null_count:,} ({percentage:.2f}%)")
    else:
        print(f"   ✅ No null values detected")
    
    return df

print("✅ Funções SCD Tipo 2 carregadas com sucesso")

## Importação dos dados da camada Bronze
Aqui temos algumas opções: 
- Importar do storage do Data Lake (landing zone);
- Importar de um shortcut;
- Importar da Tabela (Table) propagada do Lakehouse no schema broze;

Nos exemplos abaixo usaremos a terceira opção por simplicidade. 


In [8]:
# Usando Variaveis para parametrizar os codigos:

INPUT_LAYER = "bronze"
WORKSPACE_NAME = spark.conf.get("trident.workspace.name")
LAKEHOUSE_NAME = spark.conf.get("trident.lakehouse.name")
OUTPUT_LAYER = "silver"

# Diretorio de saida completo
OUTPUT_PATH = f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_NAME}.Lakehouse/Tables/{OUTPUT_LAYER}"
# Diretorio de saida curto
# Tables/{OUTPUT_LAYER}/olist_customers_dataset.csv


StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 9, Finished, Available, Finished)

In [9]:
customers_df = spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{INPUT_LAYER}.olist_customers_dataset")
geolocation_df = spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{INPUT_LAYER}.olist_geolocation_dataset")
order_items_df = spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{INPUT_LAYER}.olist_order_items_dataset")
order_payments_df = spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{INPUT_LAYER}.olist_order_payments_dataset")
order_reviews_df = spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{INPUT_LAYER}.olist_order_reviews_dataset")
orders_df = spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{INPUT_LAYER}.olist_orders_dataset")
products_df = spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{INPUT_LAYER}.olist_products_dataset")
sellers_df = spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{INPUT_LAYER}.olist_sellers_dataset")
product_category_name_translation_df = spark.sql(f"SELECT * FROM {LAKEHOUSE_NAME}.{INPUT_LAYER}.product_category_name_translation")


StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 10, Finished, Available, Finished)

## Mostrar as tabelas da camada bronze
Existem algumas formas de mostrar as tabelas lidas, isso serve para qualquer dataframe.  

```python
dataframe.show()
display(dataframe)
```

## 1 : Exibição das tabelas
Vamos olhar como estão os dados das tabelas para ver quais tipos de alterações podemos realizar.  


In [10]:
# Exibir as 5 primeiras linhas e o esquema do dataframe customers_df
customers_df.show(2,vertical=True,truncate=False)
customers_df.printSchema()

StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 11, Finished, Available, Finished)

-RECORD 0----------------------------------------------------
 customer_id              | b8f26e9e2db5ca6ecf0223a40994e02b 
 customer_unique_id       | 2369cc934990a1abc5b18b5ccf8da1ba 
 customer_zip_code_prefix | 21555                            
 customer_city            | rio de janeiro                   
 customer_state           | RJ                               
-RECORD 1----------------------------------------------------
 customer_id              | 9e226cfa83b3ebd7c9aa0650697ecce2 
 customer_unique_id       | ac6c38dcc2677a2e9510f1d508723883 
 customer_zip_code_prefix | 21740                            
 customer_city            | rio de janeiro                   
 customer_state           | RJ                               
only showing top 2 rows

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nul

In [11]:
# Exibir as 5 primeiras linhas e o esquema do dataframe geolocation_df
geolocation_df.show(2,vertical=True,truncate=False)
geolocation_df.printSchema()

StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 12, Finished, Available, Finished)

-RECORD 0------------------------------------------
 geolocation_zip_code_prefix | 21330               
 geolocation_lat             | -22.88243323896361  
 geolocation_lng             | -43.358359086588976 
 geolocation_city            | rio de janeiro      
 geolocation_state           | RJ                  
-RECORD 1------------------------------------------
 geolocation_zip_code_prefix | 21330               
 geolocation_lat             | -22.88243323896361  
 geolocation_lng             | -43.358359086588976 
 geolocation_city            | rio de janeiro      
 geolocation_state           | RJ                  
only showing top 2 rows

root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: string (nullable = true)
 |-- geolocation_lng: string (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



In [12]:
# Exibir as 5 primeiras linhas e o esquema do dataframe order_items_df
order_items_df.show(2,vertical=True,truncate=False)
order_items_df.printSchema()


StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 13, Finished, Available, Finished)

-RECORD 0-----------------------------------------------
 order_id            | 8272b63d03f5f79c56e9e4120aec44ef 
 order_item_id       | 8                                
 product_id          | 05b515fdc76e888aada3c6d66c201dff 
 seller_id           | 2709af9587499e95e803a6498a5a56e9 
 shipping_limit_date | 2017-07-21 18:25:23              
 price               | 1.20                             
 freight_value       | 7.89                             
-RECORD 1-----------------------------------------------
 order_id            | 8272b63d03f5f79c56e9e4120aec44ef 
 order_item_id       | 9                                
 product_id          | 05b515fdc76e888aada3c6d66c201dff 
 seller_id           | 2709af9587499e95e803a6498a5a56e9 
 shipping_limit_date | 2017-07-21 18:25:23              
 price               | 1.20                             
 freight_value       | 7.89                             
only showing top 2 rows

root
 |-- order_id: string (nullable = true)
 |-- order_item_id

In [13]:
# Exibir as 5 primeiras linhas e o esquema do dataframe order_payments_df
order_payments_df.show(2,vertical=True,truncate=False)
order_payments_df.printSchema()

StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 14, Finished, Available, Finished)

-RECORD 0------------------------------------------------
 order_id             | 931a32b4ca3fdc36741af29ea645d8bf 
 payment_sequential   | 2                                
 payment_type         | boleto                           
 payment_installments | 1                                
 payment_value        | 94.40                            
-RECORD 1------------------------------------------------
 order_id             | d19e65e88e48b013c91b0e77c852d06e 
 payment_sequential   | 2                                
 payment_type         | debit_card                       
 payment_installments | 1                                
 payment_value        | 99.50                            
only showing top 2 rows

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)



In [14]:
# Exibir as 5 primeiras linhas e o esquema do dataframe order_reviews_df
order_reviews_df.show(2,vertical=True,truncate=False)
order_reviews_df.printSchema()

StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 15, Finished, Available, Finished)

-RECORD 0---------------------------------------------------
 review_id               | e06c059207dad93c6808dd69aad29217 
 order_id                | de6a26eab8b3a87c4d4ec2e2a8c2495e 
 review_score            | 2                                
 review_comment_title    | Farinheiro                       
 review_comment_message  | NULL                             
 review_creation_date    | 2018-06-16 00:00:00              
 review_answer_timestamp | 2018-06-18 11:08:40              
-RECORD 1---------------------------------------------------
 review_id               | 52baca75dbcbb53c69ae3e39e4632675 
 order_id                | e38ff07f7864e8fd4fd51687cba79d89 
 review_score            | 2                                
 review_comment_title    | Médio                            
 review_comment_message  | NULL                             
 review_creation_date    | 2018-07-25 00:00:00              
 review_answer_timestamp | 2018-07-28 03:09:47              
only showing top 2 rows


In [15]:
# Exibir as 5 primeiras linhas e o esquema do dataframe orders_df
orders_df.show(2,vertical=True,truncate=False)
orders_df.printSchema()


StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 16, Finished, Available, Finished)

-RECORD 0---------------------------------------------------------
 order_id                      | 7813842ae95e8c497fc0233232ae815a 
 customer_id                   | 040d94f8ba8ca26014bd6f7e8a6e0c0d 
 order_status                  | canceled                         
 order_purchase_timestamp      | 2018-08-17 20:06:36              
 order_approved_at             | NULL                             
 order_delivered_carrier_date  | NULL                             
 order_delivered_customer_date | NULL                             
 order_estimated_delivery_date | 2018-09-17 00:00:00              
-RECORD 1---------------------------------------------------------
 order_id                      | 5a14c8b3d919a4ef3f3428b0459c47b2 
 customer_id                   | 666094835d60d986eb87350b31efdcae 
 order_status                  | canceled                         
 order_purchase_timestamp      | 2017-05-29 23:53:39              
 order_approved_at             | NULL                         

In [16]:
# Exibir as 5 primeiras linhas e o esquema do dataframe products_df
products_df.show(2,vertical=True,truncate=False)
products_df.printSchema()

StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 17, Finished, Available, Finished)

-RECORD 0------------------------------------------------------
 product_id                 | a41e356c76fab66334f36de622ecbd3a 
 product_category_name      | NULL                             
 product_name_lenght        | NULL                             
 product_description_lenght | NULL                             
 product_photos_qty         | NULL                             
 product_weight_g           | 650                              
 product_length_cm          | 17                               
 product_height_cm          | 14                               
 product_width_cm           | 12                               
-RECORD 1------------------------------------------------------
 product_id                 | d8dee61c2034d6d075997acef1870e9b 
 product_category_name      | NULL                             
 product_name_lenght        | NULL                             
 product_description_lenght | NULL                             
 product_photos_qty         | NULL      

In [17]:
# Exibir as 5 primeiras linhas e o esquema do dataframe sellers_df
sellers_df.show(2,vertical=True,truncate=False)
sellers_df.printSchema()


StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 18, Finished, Available, Finished)

-RECORD 0--------------------------------------------------
 seller_id              | 392f7f2c797e4dc077e4311bde2ab8ce 
 seller_zip_code_prefix | 21210                            
 seller_city            | rio de janeiro                   
 seller_state           | RN                               
-RECORD 1--------------------------------------------------
 seller_id              | 99002261c568a84cce14d43fcffb43ea 
 seller_zip_code_prefix | 78095                            
 seller_city            | cuiaba                           
 seller_state           | MT                               
only showing top 2 rows

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [18]:
# Exibir as 5 primeiras linhas e o esquema do dataframe product_category_name_translation_df
product_category_name_translation_df.show(2,vertical=True,truncate=False)
product_category_name_translation_df.printSchema()

StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 19, Finished, Available, Finished)

-RECORD 0-----------------------------------------------
 product_category_name         | beleza_saude           
 product_category_name_english | health_beauty          
-RECORD 1-----------------------------------------------
 product_category_name         | informatica_acessorios 
 product_category_name_english | computers_accessories  
only showing top 2 rows

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



## O que modificar no schema? 

Olhando os tipos de dados de saida do printSchema podemos ver quais dados remapear: 

| Dataframe                | Column                        | Output Data Type |
|--------------------------|-------------------------------|------------------|
| products_df              | product_weight_g              | Integer          |
| products_df              | product_length_cm             | Integer          |
| products_df              | product_height_cm             | Integer          |
| products_df              | product_width_cm              | Integer          |
| orders_df                | order_purchase_timestamp      | Timestamp        |
| orders_df                | order_approved_at             | NULL             |
| orders_df                | order_delivered_carrier_date  | NULL             |
| orders_df                | order_delivered_customer_date | NULL             |
| orders_df                | order_estimated_delivery_date | Timestamp        |
| order_reviews_df        | review_score                  | Integer          |
| order_reviews_df        | review_creation_date          | Timestamp        |
| order_reviews_df        | review_answer_timestamp       | Timestamp        |
| order_payments_df        | payment_sequential            | Integer          |
| order_payments_df        | payment_installments          | Integer          |
| order_payments_df        | payment_value                 | Double           |
| order_items_df           | shipping_limit_date           | Timestamp        |
| order_items_df           | price                         | Double           |
| order_items_df           | freight_value                 | Double           |

Porque não passar os dados de zip code prefix para integer?  
Neste caso não passarei para integer devido a não fazer diferença para calculo, são dados de CEP no Brasil onde o CEP completo tem o formato xxxxx-xxx. Outro ponto aqui é que existem CEPs que começam com zero 0, o que faria que os valores ficariam incorretos, um CEP iniciado com zero nao teria 5 caracteres, e sim menos.  
Caso no futuro haja a necessidade de troca basta adicionar mais uma etapa no pipeline ou trocar na saida (PowerBI). 


## 🔄 Aplicando Transformações SCD Tipo 2 e Regras de Negócio

### 📊 Estratégia de Transformação
1. **Data Profiling**: Análise inicial da qualidade dos dados
2. **Schema Evolution**: Padronização de tipos e correção de nomes
3. **Business Rules**: Aplicação de regras específicas da Olist
4. **SCD Type 2**: Implementação de versionamento histórico
5. **Data Quality**: Validações e métricas de qualidade


### 🏗️ 1. Schema Evolution e Data Type Standardization

In [ ]:
# =============================================================================
# PRODUCTS - Schema Corrections + Business Rules
# =============================================================================
print("🔧 Processando Products com SCD Tipo 2...")

# Schema corrections e standardização
products_df = products_df.withColumn("product_weight_g", col("product_weight_g").cast(IntegerType())) \
                         .withColumnRenamed("product_name_lenght", "product_name_length") \
                         .withColumnRenamed("product_description_lenght", "product_description_length") \
                         .withColumn("product_length_cm", col("product_length_cm").cast(IntegerType())) \
                         .withColumn("product_height_cm", col("product_height_cm").cast(IntegerType())) \
                         .withColumn("product_photos_qty", col("product_photos_qty").cast(IntegerType())) \
                         .withColumn("product_width_cm", col("product_width_cm").cast(IntegerType()))

# Business Rules específicas para Products
products_df = products_df.withColumn("product_category_name", 
                                   when(col("product_category_name").isNull(), "sem_categoria")
                                   .otherwise(trim(lower(col("product_category_name"))))) \
                         .withColumn("is_valid_dimensions",
                                   when((col("product_length_cm") > 0) & 
                                        (col("product_height_cm") > 0) & 
                                        (col("product_width_cm") > 0), True)
                                   .otherwise(False)) \
                         .withColumn("volume_cm3",
                                   when(col("is_valid_dimensions") == True,
                                        col("product_length_cm") * col("product_height_cm") * col("product_width_cm"))
                                   .otherwise(None)) \
                         .withColumn("weight_to_volume_ratio",
                                   when((col("volume_cm3").isNotNull()) & (col("volume_cm3") > 0),
                                        col("product_weight_g") / col("volume_cm3"))
                                   .otherwise(None))

# Aplicar SCD Tipo 2
products_df = add_scd2_columns(products_df, ["product_id"], 
                              exclude_cols=["effective_date", "end_date", "is_current", "record_hash", "change_reason"])
products_df = apply_data_quality_checks(products_df, "products")

# =============================================================================
# ORDERS - Enhanced Business Logic
# =============================================================================
print("🔧 Processando Orders com SCD Tipo 2...")

# Schema corrections
orders_df = orders_df.withColumn("order_purchase_timestamp", col("order_purchase_timestamp").cast(TimestampType())) \
                     .withColumn("order_approved_at", col("order_approved_at").cast(TimestampType())) \
                     .withColumn("order_delivered_carrier_date", col("order_delivered_carrier_date").cast(TimestampType())) \
                     .withColumn("order_delivered_customer_date", col("order_delivered_customer_date").cast(TimestampType())) \
                     .withColumn("order_estimated_delivery_date", col("order_estimated_delivery_date").cast(TimestampType()))

# Aplicar business rules específicas
orders_df = apply_business_rules_orders(orders_df)

# Métricas de performance avançadas
orders_df = orders_df.withColumn("processing_time_hours",
                                when(col("order_approved_at").isNotNull() & col("order_purchase_timestamp").isNotNull(),
                                     (col("order_approved_at").cast("long") - col("order_purchase_timestamp").cast("long")) / 3600)
                                .otherwise(None)) \
                     .withColumn("fulfillment_time_days",
                                when(col("order_delivered_carrier_date").isNotNull() & col("order_approved_at").isNotNull(),
                                     datediff(col("order_delivered_carrier_date"), col("order_approved_at")))
                                .otherwise(None)) \
                     .withColumn("delivery_performance_score",
                                when(col("is_on_time_delivery") == True, 100)
                                .when(col("delivery_days") <= 30, 80)
                                .when(col("delivery_days") <= 60, 60)
                                .otherwise(40))

# Aplicar SCD Tipo 2
orders_df = add_scd2_columns(orders_df, ["order_id"])
orders_df = apply_data_quality_checks(orders_df, "orders")

# =============================================================================
# CUSTOMERS - Enhanced with Geolocation Intelligence
# =============================================================================
print("🔧 Processando Customers com SCD Tipo 2...")

# Aplicar business rules específicas
customers_df = apply_business_rules_customers(customers_df)

# Enriquecimento com inteligência geográfica
customers_df = customers_df.withColumn("region",
                                     when(col("customer_state").isin(["SP", "RJ", "MG", "ES"]), "Sudeste")
                                     .when(col("customer_state").isin(["RS", "SC", "PR"]), "Sul")
                                     .when(col("customer_state").isin(["GO", "MT", "MS", "DF"]), "Centro-Oeste")
                                     .when(col("customer_state").isin(["BA", "SE", "AL", "PE", "PB", "RN", "CE", "PI", "MA"]), "Nordeste")
                                     .when(col("customer_state").isin(["AM", "RR", "AP", "PA", "TO", "RO", "AC"]), "Norte")
                                     .otherwise("Desconhecido")) \
                           .withColumn("metro_area",
                                     when(col("customer_city").isin(["SAO PAULO", "GUARULHOS", "CAMPINAS", "SAO BERNARDO DO CAMPO"]), "Grande São Paulo")
                                     .when(col("customer_city").isin(["RIO DE JANEIRO", "NITEROI", "DUQUE DE CAXIAS"]), "Grande Rio")
                                     .when(col("customer_city").isin(["BELO HORIZONTE", "CONTAGEM", "BETIM"]), "Grande BH")
                                     .otherwise("Outras"))

# Aplicar SCD Tipo 2
customers_df = add_scd2_columns(customers_df, ["customer_id"])
customers_df = apply_data_quality_checks(customers_df, "customers")

print("✅ Schema Evolution e Business Rules aplicadas com sucesso!")

StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 35, Finished, Available, Finished)

### 🔄 2. Transformações das Demais Entidades

In [ ]:
# =============================================================================
# ORDER REVIEWS - Sentiment Analysis Enhancement
# =============================================================================
print("🔧 Processando Order Reviews...")

order_reviews_df = order_reviews_df.withColumn("review_score", col("review_score").cast(IntegerType())) \
                                   .withColumn("review_creation_date", col("review_creation_date").cast(TimestampType())) \
                                   .withColumn("review_answer_timestamp", col("review_answer_timestamp").cast(TimestampType()))

# Enhanced sentiment analysis
order_reviews_df = order_reviews_df.withColumn("review_sentiment_detailed",
                                              when(col("review_score") == 5, "Muito Positivo")
                                              .when(col("review_score") == 4, "Positivo") 
                                              .when(col("review_score") == 3, "Neutro")
                                              .when(col("review_score") == 2, "Negativo")
                                              .when(col("review_score") == 1, "Muito Negativo")
                                              .otherwise("Sem Avaliação")) \
                               .withColumn("has_comment",
                                          when(col("review_comment_message").isNotNull() & 
                                               (length(trim(col("review_comment_message"))) > 0), True)
                                          .otherwise(False)) \
                               .withColumn("comment_length",
                                          when(col("has_comment") == True, 
                                               length(trim(col("review_comment_message"))))
                                          .otherwise(0)) \
                               .withColumn("response_time_days",
                                          when(col("review_answer_timestamp").isNotNull() & 
                                               col("review_creation_date").isNotNull(),
                                               datediff(col("review_answer_timestamp"), col("review_creation_date")))
                                          .otherwise(None))

order_reviews_df = add_scd2_columns(order_reviews_df, ["review_id"])
order_reviews_df = apply_data_quality_checks(order_reviews_df, "order_reviews")

# =============================================================================
# ORDER PAYMENTS - Financial Intelligence
# =============================================================================
print("🔧 Processando Order Payments...")

order_payments_df = order_payments_df.withColumn("payment_sequential", col("payment_sequential").cast(IntegerType())) \
                                     .withColumn("payment_installments", col("payment_installments").cast(IntegerType())) \
                                     .withColumn("payment_value", col("payment_value").cast(DoubleType()))

# Enhanced payment analytics
order_payments_df = order_payments_df.withColumn("payment_type_category",
                                                when(col("payment_type") == "credit_card", "Cartão")
                                                .when(col("payment_type") == "boleto", "Boleto")
                                                .when(col("payment_type") == "debit_card", "Cartão")
                                                .when(col("payment_type") == "voucher", "Voucher")
                                                .otherwise("Outros")) \
                                     .withColumn("is_installment_purchase",
                                                when(col("payment_installments") > 1, True)
                                                .otherwise(False)) \
                                     .withColumn("installment_category",
                                                when(col("payment_installments") == 1, "À vista")
                                                .when(col("payment_installments").between(2, 6), "Parcelado (2-6x)")
                                                .when(col("payment_installments").between(7, 12), "Parcelado (7-12x)")
                                                .when(col("payment_installments") > 12, "Longo Prazo (>12x)")
                                                .otherwise("Indefinido")) \
                                     .withColumn("installment_value",
                                                when(col("payment_installments") > 0,
                                                     col("payment_value") / col("payment_installments"))
                                                .otherwise(col("payment_value")))

order_payments_df = add_scd2_columns(order_payments_df, ["order_id", "payment_sequential"])
order_payments_df = apply_data_quality_checks(order_payments_df, "order_payments")

# =============================================================================
# ORDER ITEMS - Enhanced Product Intelligence
# =============================================================================
print("🔧 Processando Order Items...")

order_items_df = order_items_df.withColumn("shipping_limit_date", col("shipping_limit_date").cast(TimestampType())) \
                               .withColumn("price", col("price").cast(DoubleType())) \
                               .withColumn("freight_value", col("freight_value").cast(DoubleType()))

# Enhanced order items analytics
order_items_df = order_items_df.withColumn("total_item_value", col("price") + col("freight_value")) \
                               .withColumn("freight_percentage", 
                                          when(col("price") > 0, (col("freight_value") / col("price")) * 100)
                                          .otherwise(0)) \
                               .withColumn("freight_category",
                                          when(col("freight_percentage") == 0, "Frete Grátis")
                                          .when(col("freight_percentage") <= 10, "Frete Baixo")
                                          .when(col("freight_percentage") <= 25, "Frete Médio")
                                          .otherwise("Frete Alto")) \
                               .withColumn("price_tier",
                                          when(col("price") <= 50, "Baixo (≤R$50)")
                                          .when(col("price") <= 200, "Médio (R$50-200)")
                                          .when(col("price") <= 500, "Alto (R$200-500)")
                                          .otherwise("Premium (>R$500)"))

order_items_df = add_scd2_columns(order_items_df, ["order_id", "order_item_id"])
order_items_df = apply_data_quality_checks(order_items_df, "order_items")

# =============================================================================
# SELLERS & GEOLOCATION - Location Intelligence
# =============================================================================
print("🔧 Processando Sellers e Geolocation...")

# Sellers with enhanced location intelligence
sellers_df = sellers_df.withColumn("seller_zip_code_prefix", 
                                 when(col("seller_zip_code_prefix").rlike("^[0-9]{5}$"), 
                                      col("seller_zip_code_prefix"))
                                 .otherwise("00000")) \
                       .withColumn("seller_city", 
                                 regexp_replace(trim(upper(col("seller_city"))), "[^A-Z0-9 ]", "")) \
                       .withColumn("seller_state", 
                                 when(col("seller_state").rlike("^[A-Z]{2}$"), 
                                      upper(trim(col("seller_state"))))
                                 .otherwise("XX")) \
                       .withColumn("seller_region",
                                 when(col("seller_state").isin(["SP", "RJ", "MG", "ES"]), "Sudeste")
                                 .when(col("seller_state").isin(["RS", "SC", "PR"]), "Sul")
                                 .when(col("seller_state").isin(["GO", "MT", "MS", "DF"]), "Centro-Oeste")
                                 .when(col("seller_state").isin(["BA", "SE", "AL", "PE", "PB", "RN", "CE", "PI", "MA"]), "Nordeste")
                                 .when(col("seller_state").isin(["AM", "RR", "AP", "PA", "TO", "RO", "AC"]), "Norte")
                                 .otherwise("Desconhecido"))

sellers_df = add_scd2_columns(sellers_df, ["seller_id"])
sellers_df = apply_data_quality_checks(sellers_df, "sellers")

# Geolocation with data quality improvements
geolocation_df = geolocation_df.withColumn("geolocation_lat", col("geolocation_lat").cast(DoubleType())) \
                               .withColumn("geolocation_lng", col("geolocation_lng").cast(DoubleType())) \
                               .withColumn("is_valid_coordinates",
                                          when((col("geolocation_lat").between(-35, 10)) & 
                                               (col("geolocation_lng").between(-75, -30)), True)
                                          .otherwise(False)) \
                               .filter(col("is_valid_coordinates") == True) \
                               .dropDuplicates(["geolocation_lat", "geolocation_lng"])

geolocation_df = add_scd2_columns(geolocation_df, ["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"])
geolocation_df = apply_data_quality_checks(geolocation_df, "geolocation")

# Product Category Translation
product_category_name_translation_df = add_scd2_columns(product_category_name_translation_df, ["product_category_name"])

print("✅ Todas as transformações SCD Tipo 2 aplicadas com sucesso!")

## Criação das colunas de data para mes,dia,ano e dia da semana.

In [35]:
from pyspark.sql.functions import year, month, dayofmonth, dayofweek

order_items_df = order_items_df.withColumn("order_year", year(col("shipping_limit_date"))) \
                     .withColumn("order_month", month(col("shipping_limit_date"))) \
                     .withColumn("order_day", dayofmonth(col("shipping_limit_date"))) \
                     .withColumn("order_weekday", dayofweek(col("shipping_limit_date")))

order_reviews_df = order_reviews_df.withColumn("order_review_year", year(col("review_creation_date")))\
                                    .withColumn("order_review_month", month(col("review_creation_date")))\
                                    .withColumn("order_review_day", dayofmonth(col("review_creation_date")))\
                                    .withColumn("review_answer_year", year(col("review_answer_timestamp")))\
                                    .withColumn("review_answer_month", month(col("review_answer_timestamp")))\
                                    .withColumn("review_answer_day", dayofmonth(col("review_answer_timestamp")))

orders_df = orders_df.withColumn("order_purchase_year", year(col("order_purchase_timestamp")))\
                        .withColumn("order_purchase_month", month(col("order_purchase_timestamp")))\
                        .withColumn("order_purchase_day", dayofmonth(col("order_purchase_timestamp")))\
                        .withColumn("order_approved_year", year(col("order_approved_at")))\
                        .withColumn("order_approved_month", month(col("order_approved_at")))\
                        .withColumn("order_approved_day", dayofmonth(col("order_approved_at")))\
                        .withColumn("order_delivered_carrier_year", year(col("order_delivered_carrier_date")))\
                        .withColumn("order_delivered_carrier_month", month(col("order_delivered_carrier_date")))\
                        .withColumn("order_delivered_carrier_day", dayofmonth(col("order_delivered_carrier_date")))\
                        .withColumn("order_delivered_customer_year", year(col("order_delivered_customer_date")))\
                        .withColumn("order_delivered_customer_month", month(col("order_delivered_customer_date")))\
                        .withColumn("order_delivered_customer_day", dayofmonth(col("order_delivered_customer_date")))\
                        .withColumn("order_estimated_delivery_year", year(col("order_estimated_delivery_date")))\
                        .withColumn("order_estimated_delivery_month", month(col("order_estimated_delivery_date")))\
                        .withColumn("order_estimated_delivery_day", dayofmonth(col("order_estimated_delivery_date")))\








StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 36, Finished, Available, Finished)

## Remoção de Valores Nulos

In [36]:
customers_df = customers_df.fillna({"customer_zip_code_prefix": "0", 
                                    "customer_city": "Unknown",
                                    "customer_state": "Unknown",
                                    "customer_unique_id": "Unknown",
                                    "customer_id": "Unknown"
                                    })

geolocation_df = geolocation_df.fillna(
    {
        "geolocation_zip_code_prefix": "0",
        "geolocation_lat": "0",
        "geolocation_lng": "0",
        "geolocation_city": "Unknown",
        "geolocation_state": "Unknown"
    }
)

order_items_df = order_items_df.fillna({
    "order_id": "Unknown",
    "order_item_id": "0",
    "product_id": "Unknown",
    "seller_id": "Unknown",
    "shipping_limit_date": "Unknown",
    "price": 0.0,
    "freight_value": 0.0
})

order_payments_df = order_payments_df.fillna(
    {
        "order_id": "Unknown",
        "payment_sequential": 0,
        "payment_type": "Unknown",
        "payment_installments": 0,
        "payment_value": 0.0
})

order_reviews_df = order_reviews_df.fillna(
    {
        "review_id": "Unknown",
        "order_id": "Unknown",
        "review_score": 0,
        "review_comment_title": "Unknown",
        "review_comment_message": "Unknown",
        "review_creation_date": "Unknown",
        "review_answer_timestamp": "Unknown"
})


orders_df = orders_df.fillna(
    {
        "order_id": "Unknown",
        "customer_id": "Unknown",
        "order_status": "Unknown",
        "order_purchase_timestamp": "Unknown",
        "order_approved_at": "Unknown",
        "order_delivered_carrier_date": "Unknown",
        "order_delivered_customer_date": "Unknown",
        "order_estimated_delivery_date": "Unknown",
        "order_purchase_year": "Unknown",
        "order_purchase_month": "Unknown",
        "order_purchase_day": "Unknown",
        "order_approved_year": "Unknown",
        "order_approved_month": "Unknown",
        "order_approved_day": "Unknown",
        "order_delivered_carrier_year": "Unknown",
        "order_delivered_carrier_month": "Unknown",
        "order_delivered_carrier_day": "Unknown",
        "order_delivered_customer_year": "Unknown",
        "order_delivered_customer_month": "Unknown",
        "order_delivered_customer_day": "Unknown",
        "order_estimated_delivery_year": "Unknown",
        "order_estimated_delivery_month": "Unknown",
        "order_estimated_delivery_day": "Unknown",

})

products_df = products_df.fillna(
{
    "product_id": "Unknown",
    "product_category_name": "Unknown",
    "product_name_length": "0",
    "product_description_length": "0",
    "product_photos_qty": "0",
    "product_weight_g": 0,
    "product_length_cm": 0,
    "product_height_cm": 0,
    "product_width_cm": 0

})

sellers_df = sellers_df.fillna(
{
    "seller_id": "Unknown",
    "seller_zip_code_prefix": "0",
    "seller_city": "Unknown",
    "seller_state": "Unknown"
})

product_category_name_translation_df = product_category_name_translation_df.fillna(
 {
     "product_category_name": "Unknown",
     "product_category_name_english": "Unknown"
 })   

StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 37, Finished, Available, Finished)

## 💾 Salvamento Otimizado com SCD Tipo 2

### 🎯 Estratégia de Salvamento
- **Delta Lake**: Formato otimizado para ACID transactions
- **Particionamento**: Por effective_date para otimizar queries temporais  
- **Z-Ordering**: Para otimizar performance de consultas por chaves de negócio
- **Versionamento**: Controle de versões automático do Delta Lake
- **Metadata**: Enriquecimento com informações de lineage


In [ ]:
def save_to_silver_scd2(df, table_name, partition_cols=None, zorder_cols=None):
    """
    Salva DataFrame na camada Silver com otimizações SCD Tipo 2
    
    Args:
        df: DataFrame para salvar
        table_name: Nome da tabela
        partition_cols: Colunas para particionamento
        zorder_cols: Colunas para Z-Ordering
    """
    print(f"💾 Salvando {table_name} com otimizações SCD Tipo 2...")
    
    # Path da tabela
    table_path = f"{OUTPUT_PATH}/{table_name}_silver"
    
    # Configuração do writer
    writer = df.coalesce(4).write.format("delta").mode("overwrite")
    
    # Aplicar particionamento se especificado
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    
    # Salvar
    writer.option("path", table_path).save()
    
    # Aplicar Z-Ordering se especificado (requer Delta Lake)
    if zorder_cols:
        try:
            spark.sql(f"OPTIMIZE delta.`{table_path}` ZORDER BY ({', '.join(zorder_cols)})")
            print(f"  ✅ Z-Ordering aplicado em: {', '.join(zorder_cols)}")
        except Exception as e:
            print(f"  ⚠️ Z-Ordering não aplicado: {str(e)}")
    
    # Estatísticas finais
    record_count = df.count()
    current_records = df.filter(col("is_current") == True).count() if "is_current" in df.columns else record_count
    
    print(f"  📊 {table_name}: {record_count:,} total records, {current_records:,} current records")
    print(f"  📂 Salvo em: {table_path}")

print("✅ Função de salvamento SCD2 carregada")

In [ ]:
# =============================================================================
# SALVAMENTO OTIMIZADO COM SCD TIPO 2
# =============================================================================
print("🚀 Iniciando processo de salvamento otimizado...")

# 1. CUSTOMERS - Particionado por região para otimizar análises geográficas
save_to_silver_scd2(customers_df, "customers", 
                    partition_cols=["region"], 
                    zorder_cols=["customer_id", "customer_unique_id"])

# 2. ORDERS - Particionado por ano/mês para análises temporais
orders_df_with_partition = orders_df.withColumn("year_month", 
                                               concat(year(col("order_purchase_timestamp")), 
                                                     lit("-"), 
                                                     lpad(month(col("order_purchase_timestamp")), 2, "0")))

save_to_silver_scd2(orders_df_with_partition, "orders",
                    partition_cols=["year_month"],
                    zorder_cols=["order_id", "customer_id"])

# 3. PRODUCTS - Particionado por categoria para análises de produto
save_to_silver_scd2(products_df, "products",
                    partition_cols=["product_category_name"],
                    zorder_cols=["product_id"])

# 4. ORDER_ITEMS - Particionado por ano/mês do shipping_limit_date
order_items_df_with_partition = order_items_df.withColumn("year_month",
                                                         concat(year(col("shipping_limit_date")),
                                                               lit("-"),
                                                               lpad(month(col("shipping_limit_date")), 2, "0")))

save_to_silver_scd2(order_items_df_with_partition, "order_items",
                    partition_cols=["year_month"],
                    zorder_cols=["order_id", "product_id", "seller_id"])

# 5. ORDER_PAYMENTS - Particionado por tipo de pagamento
save_to_silver_scd2(order_payments_df, "order_payments",
                    partition_cols=["payment_type_category"],
                    zorder_cols=["order_id", "payment_sequential"])

# 6. ORDER_REVIEWS - Particionado por sentiment para análises de satisfação
save_to_silver_scd2(order_reviews_df, "order_reviews",
                    partition_cols=["review_sentiment_detailed"],
                    zorder_cols=["review_id", "order_id"])

# 7. SELLERS - Particionado por região
save_to_silver_scd2(sellers_df, "sellers",
                    partition_cols=["seller_region"],
                    zorder_cols=["seller_id"])

# 8. GEOLOCATION - Sem particionamento (dados geográficos únicos)
save_to_silver_scd2(geolocation_df, "geolocation",
                    zorder_cols=["geolocation_zip_code_prefix"])

# 9. PRODUCT_CATEGORY_TRANSLATION - Tabela pequena, sem otimizações especiais
save_to_silver_scd2(product_category_name_translation_df, "product_category_translation")

print("\n" + "="*80)
print("🎉 TRANSFORMAÇÕES SILVER LAYER COM SCD TIPO 2 CONCLUÍDAS!")
print("="*80)
print()

# Estatísticas finais do processamento
print("📈 RESUMO DO PROCESSAMENTO:")
print(f"   🔸 Customers: {customers_df.count():,} registros")
print(f"   🔸 Orders: {orders_df.count():,} registros") 
print(f"   🔸 Products: {products_df.count():,} registros")
print(f"   🔸 Order Items: {order_items_df.count():,} registros")
print(f"   🔸 Order Payments: {order_payments_df.count():,} registros")
print(f"   🔸 Order Reviews: {order_reviews_df.count():,} registros")
print(f"   🔸 Sellers: {sellers_df.count():,} registros")
print(f"   🔸 Geolocation: {geolocation_df.count():,} registros")
print()

print("🔍 RECURSOS IMPLEMENTADOS:")
print("   ✅ SCD Tipo 2 para tracking histórico")
print("   ✅ Business rules específicas da Olist")
print("   ✅ Data quality checks e validações")
print("   ✅ Enriquecimento geográfico e demográfico")
print("   ✅ Métricas de performance e satisfação")
print("   ✅ Otimizações Delta Lake (partitioning + z-ordering)")
print("   ✅ Lineage e auditoria completa")
print()

print("🎯 PRÓXIMOS PASSOS:")
print("   1. Execute o notebook 03-GoldTransformationsDim.ipynb")
print("   2. Execute o notebook 04-GoldTransformationsFact.ipynb") 
print("   3. Execute o notebook 05-GoldOptimizations.ipynb")
print("   4. Explore os dados no Power BI ou Data Agents")


StatementMeta(, 9379e2ec-9b72-4416-88e8-96c18af1123b, 39, Finished, Available, Finished)